In [1]:
import torch
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))
from Models.Baseline_Model.GPT2_Baseline import GPT2_Baseline
from Models.Baseline_Model.train_Baseline import train_loop, estimate_loss
from Models.Configs import TrainConfig, BaselineConfig
from Datasets.DataLoader import CombinedBinDataLoader


device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to mps


In [2]:
#Load the hyperparameters, scheduler, etc... required for training
model_config = BaselineConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = GPT2_Baseline(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


/Users/davidheimowitz/.pyenv/versions/3.12.5/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


RuntimeError: Dynamo is not supported on Python 3.12+

In [ ]:
#Only run this next line once between all models to download and preprocess the dataset. Comment it out afterwards


# CombinedBinDataLoader.fetch_dataset(str(PROJECT_ROOT) + '/Datasets/fineweb_1B.bin', 1000000000)


In [ ]:
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    str(PROJECT_ROOT) + '/Datasets/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

In [ ]:
#Actually train the model
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config)

In [ ]:
message = model.infer("""One, two, three, """, 100, .8, 50)
print(message)

In [ ]:
#Save the model's state after training
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_baseline_step_3623.pt"
torch.save({
            "step": 3623 ,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history
        }, path)

In [ ]:
loss_fwd, ppl = estimate_loss(model, val_loader, device, 160)
print(f"Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f}")